# False-Positive Audit Explorer

This notebook helps inspect the audited `DeepLabV3+` false-positive banks in `results/hard_negative_audit/deeplabv3plus_tsbank_round1`.

Use it to:
- browse review cards by `bank / layer1 / layer2`
- compare numeric feature distributions for `hard_fp / ambiguous / noise`
- inspect candidate rule-based filters before running a new auto-filtered-bank experiment
- summarize repeatable visual patterns for paper-facing analysis


In [ ]:
from __future__ import annotations

import ast
import csv
import html
import json
import random
import sys
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display
from PIL import Image

WORKDIR = Path.cwd().resolve()
if (WORKDIR / "results").exists() and (WORKDIR / "notebooks").exists():
    ROOT = WORKDIR
elif WORKDIR.name == "notebooks" and (WORKDIR.parent / "results").exists():
    ROOT = WORKDIR.parent
else:
    ROOT = Path("..").resolve()

sys.path.insert(0, str(ROOT))

from dataset import CrackDataset

AUDIT_DIR = ROOT / "results" / "hard_negative_audit" / "deeplabv3plus_tsbank_round1"
AUDIT_CSV = AUDIT_DIR / "audit_samples.csv"
BANK_OVERVIEW_CSV = AUDIT_DIR / "bank_overview.csv"
DEFAULT_BANK = "uav_hard_negatives_deeplabv3plus_plain_360_train_thr080_ts"

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 11


In [ ]:
NUMERIC_FIELDS = [
    "score",
    "component_area",
    "component_mean_prob",
    "component_aspect_ratio",
    "component_fill_ratio",
    "crop_size",
    "crop_left",
    "crop_top",
]
DERIVED_FIELDS = [
    "edge_distance",
    "span_ratio",
    "bbox_width_frac",
    "bbox_height_frac",
    "touches_edge",
]

LAYER1_ORDER = ["hard_fp", "ambiguous", "noise", "bad_crop", "gt_issue", "<blank>"]
LAYER2_ORDER = [
    "pavement_edge",
    "shadow_dark_stripe",
    "line_like_texture",
    "surface_boundary",
    "debris_object",
    "other_target_clutter",
    "<blank>",
]
LAYER1_COLORS = {
    "hard_fp": "#2f7d32",
    "ambiguous": "#c58b00",
    "noise": "#b33a3a",
    "bad_crop": "#6c757d",
    "gt_issue": "#5b5f97",
    "<blank>": "#adb5bd",
}


def load_csv(path: Path) -> list[dict]:
    with path.open(encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def parse_bbox(text: str) -> tuple[int, int, int, int]:
    x, y, w, h = ast.literal_eval(text)
    return int(x), int(y), int(w), int(h)


def format_float(value: float, digits: int = 4) -> str:
    return f"{value:.{digits}f}"


def make_html_table(headers: list[str], rows: list[list[str]]) -> str:
    table_rows = []
    table_rows.append("<table border='1' cellspacing='0' cellpadding='6'>")
    header_html = "".join(f"<th>{html.escape(str(h))}</th>" for h in headers)
    table_rows.append(f"<thead><tr>{header_html}</tr></thead>")
    table_rows.append("<tbody>")
    for row in rows:
        row_html = "".join(f"<td>{html.escape(str(cell))}</td>" for cell in row)
        table_rows.append(f"<tr>{row_html}</tr>")
    table_rows.append("</tbody></table>")
    return "\n".join(table_rows)


def display_table(headers: list[str], rows: list[list[str]]):
    display(HTML(make_html_table(headers, rows)))


def build_audit_rows():
    audit_rows_raw = load_csv(AUDIT_CSV)
    bank_overview_rows = load_csv(BANK_OVERVIEW_CSV)
    bank_overview_by_label = {row["bank_label"]: row for row in bank_overview_rows}

    manifest_by_bank = {}
    datasets = {}
    for bank_label, overview in bank_overview_by_label.items():
        bank_root = Path(overview["bank_root"])
        manifest_entries = load_jsonl(bank_root / "manifest.jsonl")
        manifest_by_bank[bank_label] = {
            entry["image_rel"]: entry for entry in manifest_entries
        }

        dataset_key = (overview["dataset_root"], overview["split"])
        if dataset_key not in datasets:
            datasets[dataset_key] = CrackDataset(
                root=overview["dataset_root"],
                split=overview["split"],
                img_size=360,
            )

    audit_rows = []
    for row in audit_rows_raw:
        entry = dict(row)
        for field in NUMERIC_FIELDS:
            entry[field] = float(entry[field])
        entry["rank_overall"] = int(entry["rank_overall"])
        entry["component_bbox_xywh"] = parse_bbox(entry["component_bbox_xywh"])
        entry["layer1"] = entry["layer1_review_label"].strip() or "<blank>"
        entry["layer2"] = entry["layer2_review_taxonomy"].strip() or "<blank>"
        overview = bank_overview_by_label[entry["bank_label"]]
        entry["bank_root"] = Path(overview["bank_root"])
        manifest_entry = manifest_by_bank[entry["bank_label"]][entry["image_rel"]]
        entry["manifest"] = manifest_entry
        entry["image_path"] = entry["bank_root"] / entry["image_rel"]
        entry["mask_path"] = entry["bank_root"] / manifest_entry["mask_rel"]
        entry["review_card_path"] = AUDIT_DIR / entry["review_card_rel"]
        x, y, w, h = entry["component_bbox_xywh"]
        entry["bbox_x"] = x
        entry["bbox_y"] = y
        entry["bbox_width"] = w
        entry["bbox_height"] = h
        entry["bbox_max_side"] = max(w, h)

        dataset_key = (overview["dataset_root"], overview["split"])
        raw_image, raw_mask = datasets[dataset_key].get_raw(int(manifest_entry["source_index"]))
        image_h, image_w = raw_image.shape[:2]
        entry["image_height"] = image_h
        entry["image_width"] = image_w
        entry["edge_distance"] = min(
            x,
            y,
            image_w - (x + w),
            image_h - (y + h),
        )
        entry["bbox_width_frac"] = w / image_w
        entry["bbox_height_frac"] = h / image_h
        entry["span_ratio"] = max(entry["bbox_width_frac"], entry["bbox_height_frac"])
        entry["touches_edge"] = int(entry["edge_distance"] <= 0)
        entry["raw_mask_foreground_pixels"] = int((raw_mask > 0).sum())
        audit_rows.append(entry)

    return audit_rows, bank_overview_rows


audit_rows, bank_overview_rows = build_audit_rows()
bank_labels = [row["bank_label"] for row in bank_overview_rows]


In [ ]:
print(f"Loaded {len(audit_rows)} audited rows across {len(bank_overview_rows)} banks.")
print(f"Audit directory: {AUDIT_DIR}")

layer1_counts = Counter(row["layer1"] for row in audit_rows)
layer2_counts = Counter(row["layer2"] for row in audit_rows if row["layer2"] != "<blank>")

display_table(
    headers=["bank_label", "num_entries", "audited", "component_area_median", "mean_prob_median"],
    rows=[
        [
            row["bank_label"],
            row["num_entries"],
            row["num_selected_for_audit"],
            format_float(float(row["component_area_median"]), 1),
            format_float(float(row["component_mean_prob_median"]), 4),
        ]
        for row in bank_overview_rows
    ],
)

display_table(
    headers=["layer1", "count"],
    rows=[[label, layer1_counts.get(label, 0)] for label in LAYER1_ORDER if layer1_counts.get(label, 0)],
)

display_table(
    headers=["layer2", "count"],
    rows=[[label, layer2_counts.get(label, 0)] for label in LAYER2_ORDER if layer2_counts.get(label, 0)],
)


## 1. Overview Charts

These plots give a quick macro view before drilling into individual review cards.


In [ ]:
def plot_counter(counter: Counter, title: str, order: list[str], color_map: dict[str, str] | None = None):
    labels = [label for label in order if counter.get(label, 0)]
    values = [counter[label] for label in labels]
    colors = None
    if color_map is not None:
        colors = [color_map.get(label, "#4c72b0") for label in labels]
    plt.figure(figsize=(10, 4))
    plt.bar(labels, values, color=colors)
    plt.title(title)
    plt.ylabel("count")
    plt.xticks(rotation=20)
    plt.show()


def plot_layer1_by_bank(rows: list[dict]):
    grouped = defaultdict(Counter)
    for row in rows:
        grouped[row["bank_label"]][row["layer1"]] += 1

    banks = list(grouped.keys())
    x = np.arange(len(banks))
    width = 0.75
    bottom = np.zeros(len(banks))

    plt.figure(figsize=(12, 5))
    for label in LAYER1_ORDER:
        values = [grouped[bank].get(label, 0) for bank in banks]
        if not any(values):
            continue
        plt.bar(
            x,
            values,
            width=width,
            bottom=bottom,
            label=label,
            color=LAYER1_COLORS.get(label, "#4c72b0"),
        )
        bottom += np.array(values)

    plt.xticks(x, banks, rotation=20, ha="right")
    plt.ylabel("audited sample count")
    plt.title("Layer-1 review distribution by bank")
    plt.legend()
    plt.tight_layout()
    plt.show()


plot_counter(layer1_counts, "Overall layer-1 distribution", LAYER1_ORDER, LAYER1_COLORS)
plot_counter(layer2_counts, "Overall layer-2 distribution", LAYER2_ORDER)
plot_layer1_by_bank(audit_rows)


## 2. Example Browser

Use `select_rows(...)` to slice the audit package and `display_grid(...)` to inspect either review cards or the raw crop images.


In [ ]:
def _matches(value: str, query):
    if query is None:
        return True
    if isinstance(query, (list, tuple, set)):
        return value in query
    return value == query


def select_rows(
    rows: list[dict],
    bank_label=None,
    layer1=None,
    layer2=None,
    rank_bucket=None,
    sort_by: str = "score",
    descending: bool = True,
):
    subset = []
    for row in rows:
        if not _matches(row["bank_label"], bank_label):
            continue
        if not _matches(row["layer1"], layer1):
            continue
        if not _matches(row["layer2"], layer2):
            continue
        if not _matches(row["rank_bucket"], rank_bucket):
            continue
        subset.append(row)
    subset.sort(key=lambda row: row[sort_by], reverse=descending)
    return subset


def default_caption(row: dict) -> str:
    return (
        f"{row['layer1']} | {row['layer2']}\n"
        f"score={row['score']:.1f} area={int(row['component_area'])} fill={row['component_fill_ratio']:.2f}"
    )


def display_grid(
    rows: list[dict],
    *,
    image_field: str = "review_card_path",
    title: str | None = None,
    max_items: int = 12,
    cols: int = 4,
    randomize: bool = False,
    seed: int = 7,
    caption_fn=default_caption,
):
    if not rows:
        print("No rows matched this filter.")
        return

    subset = list(rows)
    if randomize:
        rng = random.Random(seed)
        rng.shuffle(subset)
    subset = subset[:max_items]

    cols = min(cols, len(subset))
    nrows = int(np.ceil(len(subset) / cols))
    fig, axes = plt.subplots(nrows, cols, figsize=(4.2 * cols, 3.7 * nrows))
    axes = np.atleast_1d(axes).reshape(nrows, cols)

    for ax in axes.ravel():
        ax.axis("off")

    for ax, row in zip(axes.ravel(), subset):
        image = np.array(Image.open(row[image_field]).convert("RGB"))
        ax.imshow(image)
        ax.set_title(caption_fn(row), fontsize=9)
        ax.axis("off")

    if title:
        fig.suptitle(title, fontsize=14)
        fig.tight_layout(rect=(0, 0, 1, 0.96))
    else:
        fig.tight_layout()
    plt.show()


ts_rows = select_rows(audit_rows, bank_label=DEFAULT_BANK)
display_grid(
    select_rows(ts_rows, layer1="hard_fp"),
    title="Calibrated TS-bank: hard_fp examples",
    max_items=8,
)
display_grid(
    select_rows(ts_rows, layer1="noise"),
    title="Calibrated TS-bank: noise examples",
    max_items=8,
)


## 3. Numeric Feature Comparison

These plots are useful for checking whether simple rules such as `min_area` or `max_fill_ratio` might separate useful hard negatives from obvious noise.


In [ ]:
def plot_histograms(
    rows: list[dict],
    fields: list[str],
    *,
    bank_label: str = DEFAULT_BANK,
    labels=("hard_fp", "ambiguous", "noise"),
):
    subset = select_rows(rows, bank_label=bank_label)
    fig, axes = plt.subplots(len(fields), 1, figsize=(10, 3.6 * len(fields)))
    axes = np.atleast_1d(axes)

    for ax, field in zip(axes, fields):
        for label in labels:
            values = [row[field] for row in subset if row["layer1"] == label]
            if not values:
                continue
            ax.hist(values, bins=14, alpha=0.45, label=label, color=LAYER1_COLORS.get(label))
        if field == "component_area":
            ax.set_xscale("log")
        ax.set_title(f"{bank_label} | {field}")
        ax.legend()
    fig.tight_layout()
    plt.show()


def scatter_by_label(
    rows: list[dict],
    x_field: str,
    y_field: str,
    *,
    bank_label: str = DEFAULT_BANK,
    labels=("hard_fp", "ambiguous", "noise"),
):
    subset = select_rows(rows, bank_label=bank_label)
    plt.figure(figsize=(8, 6))
    for label in labels:
        x = [row[x_field] for row in subset if row["layer1"] == label]
        y = [row[y_field] for row in subset if row["layer1"] == label]
        if not x:
            continue
        plt.scatter(x, y, label=label, alpha=0.8, s=40, color=LAYER1_COLORS.get(label))
    if x_field == "component_area":
        plt.xscale("log")
    plt.xlabel(x_field)
    plt.ylabel(y_field)
    plt.title(f"{bank_label}: {x_field} vs {y_field}")
    plt.legend()
    plt.show()


plot_histograms(
    audit_rows,
    ["component_area", "component_fill_ratio", "component_aspect_ratio", "component_mean_prob"],
)
scatter_by_label(audit_rows, "component_area", "component_fill_ratio")
scatter_by_label(audit_rows, "component_area", "component_mean_prob")


## 4. Derived Features And Taxonomy Cross-Analysis

These cells add two useful analysis layers:

- derived features that can be computed from the existing manifest without rerunning the model
- `layer2 taxonomy × layer1 label` cross-tabs to check whether certain nuisance types are usually useful hard false positives or mostly noise

Notes:
- `edge_distance` measures how close the component bbox is to the image border
- `span_ratio` measures how much of the image width or height the component spans
- `compactness` is not available here because the manifest does not store the component perimeter or binary component mask


In [ ]:
def summarize_fields_by_layer1(
    rows: list[dict],
    fields: list[str],
    *,
    bank_label: str = DEFAULT_BANK,
    labels=("hard_fp", "ambiguous", "noise"),
):
    subset = select_rows(rows, bank_label=bank_label)
    headers = ["layer1", "count"] + [f"{field}_median" for field in fields]
    table_rows = []
    for label in labels:
        label_rows = [row for row in subset if row["layer1"] == label]
        if not label_rows:
            continue
        row_values = [label, len(label_rows)]
        for field in fields:
            values = [row[field] for row in label_rows]
            row_values.append(format_float(float(np.median(values)), 3))
        table_rows.append(row_values)
    display_table(headers, table_rows)


def taxonomy_crosstab(
    rows: list[dict],
    *,
    bank_label: str | None = None,
    normalize: bool = False,
    include_blank: bool = False,
):
    subset = select_rows(rows, bank_label=bank_label)
    grouped = defaultdict(Counter)
    for row in subset:
        if not include_blank and row["layer2"] == "<blank>":
            continue
        grouped[row["layer2"]][row["layer1"]] += 1

    labels = [label for label in LAYER1_ORDER if label != "<blank>"]
    table_rows = []
    for taxonomy in [item for item in LAYER2_ORDER if include_blank or item != "<blank>"]:
        counter = grouped.get(taxonomy)
        if not counter:
            continue
        total = sum(counter.values())
        row = [taxonomy, total]
        for label in labels:
            value = counter.get(label, 0)
            if normalize and total:
                row.append(format_float(value / total, 3))
            else:
                row.append(value)
        table_rows.append(row)

    headers = ["layer2", "total"] + labels
    display_table(headers, table_rows)


def plot_taxonomy_stacked(rows: list[dict], *, bank_label: str | None = None):
    subset = select_rows(rows, bank_label=bank_label)
    grouped = defaultdict(Counter)
    for row in subset:
        if row["layer2"] == "<blank>":
            continue
        grouped[row["layer2"]][row["layer1"]] += 1

    taxonomies = [taxonomy for taxonomy in LAYER2_ORDER if taxonomy != "<blank>" and taxonomy in grouped]
    x = np.arange(len(taxonomies))
    bottom = np.zeros(len(taxonomies))

    plt.figure(figsize=(11, 5))
    for label in ["hard_fp", "ambiguous", "noise", "bad_crop", "gt_issue"]:
        values = [grouped[taxonomy].get(label, 0) for taxonomy in taxonomies]
        if not any(values):
            continue
        plt.bar(
            x,
            values,
            bottom=bottom,
            color=LAYER1_COLORS.get(label),
            label=label,
        )
        bottom += np.array(values)

    plt.xticks(x, taxonomies, rotation=20, ha="right")
    plt.ylabel("count")
    plt.title(f"Layer-1 distribution within each layer-2 taxonomy | bank={bank_label or 'all'}")
    plt.legend()
    plt.tight_layout()
    plt.show()


plot_histograms(
    audit_rows,
    ["component_area", "component_fill_ratio", "edge_distance", "span_ratio"],
)
summarize_fields_by_layer1(
    audit_rows,
    ["component_area", "component_fill_ratio", "edge_distance", "span_ratio"],
)

taxonomy_crosstab(audit_rows, bank_label=DEFAULT_BANK, normalize=False)
taxonomy_crosstab(audit_rows, bank_label=DEFAULT_BANK, normalize=True)
plot_taxonomy_stacked(audit_rows, bank_label=DEFAULT_BANK)


## 5. Span-Ratio Sweep For Auto-Filter Ideas

This extra cell compares `span_ratio` thresholds against the current `area >= 1200` baseline rule.

Why this matters:
- `span_ratio` is more semantic than area alone because it measures how much of the image the predicted component spans
- a larger `span_ratio` often corresponds to road-edge or material-boundary structures that the model mistakes for cracks
- the sweep helps choose a threshold before exporting an `auto-filtered` bank


In [ ]:
def summarize_rule_counts(rows: list[dict], *, keep_fn, bank_label: str = DEFAULT_BANK):
    subset = select_rows(rows, bank_label=bank_label)
    kept = [row for row in subset if keep_fn(row)]
    total_hard = sum(row["layer1"] == "hard_fp" for row in subset)
    total_amb = sum(row["layer1"] == "ambiguous" for row in subset)
    total_noise = sum(row["layer1"] == "noise" for row in subset)
    kept_hard = sum(row["layer1"] == "hard_fp" for row in kept)
    kept_amb = sum(row["layer1"] == "ambiguous" for row in kept)
    kept_noise = sum(row["layer1"] == "noise" for row in kept)
    return {
        "kept_rows": len(kept),
        "hard_fp_kept": kept_hard,
        "hard_fp_total": total_hard,
        "ambiguous_kept": kept_amb,
        "ambiguous_total": total_amb,
        "noise_kept": kept_noise,
        "noise_total": total_noise,
        "noise_removed": total_noise - kept_noise,
        "hard_fp_precision": kept_hard / len(kept) if kept else 0.0,
        "hard_fp_recall": kept_hard / total_hard if total_hard else 0.0,
    }


def sweep_span_ratio(rows: list[dict], *, bank_label: str = DEFAULT_BANK, thresholds=(0.30, 0.35, 0.40, 0.45, 0.50)):
    table_rows = []
    for threshold in thresholds:
        stats = summarize_rule_counts(
            rows,
            bank_label=bank_label,
            keep_fn=lambda row, threshold=threshold: row["span_ratio"] >= threshold,
        )
        table_rows.append([
            f">= {threshold:.2f}",
            stats["kept_rows"],
            f"{stats['hard_fp_kept']} / {stats['hard_fp_total']}",
            f"{stats['ambiguous_kept']} / {stats['ambiguous_total']}",
            f"{stats['noise_removed']} / {stats['noise_total']}",
            format_float(stats["hard_fp_precision"], 3),
            format_float(stats["hard_fp_recall"], 3),
        ])
    display_table(
        headers=[
            "span_ratio rule",
            "kept rows",
            "hard_fp kept",
            "ambiguous kept",
            "noise removed",
            "hard_fp precision among kept",
            "hard_fp recall",
        ],
        rows=table_rows,
    )


def compare_area_and_span(rows: list[dict], *, bank_label: str = DEFAULT_BANK):
    configs = [
        ("area >= 1200", lambda row: row["component_area"] >= 1200),
        ("span_ratio >= 0.35", lambda row: row["span_ratio"] >= 0.35),
    ]
    table_rows = []
    for name, keep_fn in configs:
        stats = summarize_rule_counts(rows, bank_label=bank_label, keep_fn=keep_fn)
        table_rows.append([
            name,
            stats["kept_rows"],
            f"{stats['hard_fp_kept']} / {stats['hard_fp_total']}",
            f"{stats['ambiguous_kept']} / {stats['ambiguous_total']}",
            f"{stats['noise_removed']} / {stats['noise_total']}",
            format_float(stats["hard_fp_precision"], 3),
            format_float(stats["hard_fp_recall"], 3),
        ])
    display_table(
        headers=[
            "rule",
            "kept rows",
            "hard_fp kept",
            "ambiguous kept",
            "noise removed",
            "hard_fp precision among kept",
            "hard_fp recall",
        ],
        rows=table_rows,
    )


sweep_span_ratio(audit_rows, bank_label=DEFAULT_BANK)
compare_area_and_span(audit_rows, bank_label=DEFAULT_BANK)


## 6. Rule Sandbox

Use this section to test cheap geometric filters against the audited labels before exporting an `auto-filtered` bank.

The default example below evaluates a simple `component_area >= 1200` rule on the calibrated `TS-bank`.


In [ ]:
def evaluate_rule(
    rows: list[dict],
    *,
    bank_label: str = DEFAULT_BANK,
    min_area: float = 0.0,
    max_fill_ratio: float = 1.0,
    min_aspect_ratio: float = 0.0,
    max_bbox_side: float | None = None,
    max_span_ratio: float | None = None,
    min_edge_distance: float | None = None,
):
    subset = select_rows(rows, bank_label=bank_label)

    def keep(row: dict) -> bool:
        if row["component_area"] < min_area:
            return False
        if row["component_fill_ratio"] > max_fill_ratio:
            return False
        if row["component_aspect_ratio"] < min_aspect_ratio:
            return False
        if max_bbox_side is not None and row["bbox_max_side"] > max_bbox_side:
            return False
        if max_span_ratio is not None and row["span_ratio"] > max_span_ratio:
            return False
        if min_edge_distance is not None and row["edge_distance"] < min_edge_distance:
            return False
        return True

    kept = [row for row in subset if keep(row)]
    removed = [row for row in subset if not keep(row)]
    total_hard_fp = sum(row["layer1"] == "hard_fp" for row in subset)
    kept_hard_fp = sum(row["layer1"] == "hard_fp" for row in kept)
    hard_fp_precision = kept_hard_fp / len(kept) if kept else 0.0
    hard_fp_recall = kept_hard_fp / total_hard_fp if total_hard_fp else 0.0

    counts = defaultdict(lambda: {"kept": 0, "removed": 0})
    for row in kept:
        counts[row["layer1"]]["kept"] += 1
    for row in removed:
        counts[row["layer1"]]["removed"] += 1

    print(f"bank={bank_label}")
    print(
        f"rule: area >= {min_area}, fill <= {max_fill_ratio}, "
        f"aspect >= {min_aspect_ratio}, max_bbox_side <= {max_bbox_side}, "
        f"span_ratio <= {max_span_ratio}, edge_distance >= {min_edge_distance}"
    )
    print(f"kept {len(kept)} / {len(subset)} rows")
    print(f"hard_fp precision among kept rows = {hard_fp_precision:.3f}")
    print(f"hard_fp recall within audited bank = {hard_fp_recall:.3f}")

    display_table(
        headers=["layer1", "kept", "removed"],
        rows=[
            [label, counts[label]["kept"], counts[label]["removed"]]
            for label in LAYER1_ORDER
            if counts[label]["kept"] or counts[label]["removed"]
        ],
    )
    return kept, removed


kept_rows, removed_rows = evaluate_rule(audit_rows, bank_label=DEFAULT_BANK, min_area=1200)
display_grid(
    [row for row in kept_rows if row["layer1"] == "noise"],
    title="Noise that survives area >= 1200",
    max_items=6,
)
display_grid(
    [row for row in removed_rows if row["layer1"] == "hard_fp"],
    title="Hard FP lost by area >= 1200",
    max_items=6,
)
